In [2]:
import pandas as pd

expression_matrix = "Data/GSE96058_gene_expression_3273_samples_and_136_replicates_transformed.csv"
expression_df = pd.read_csv(expression_matrix, index_col=0)
expression_df.head()

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,...,F2974repl,F3006repl,F3028repl,F3057repl,F3058repl,F3085repl,F3127repl,F3135repl,F3250repl,F3265repl
5_8S_rRNA,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,...,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
5S_rRNA,4.911099,-3.321928,-3.321928,3.656393,4.190104,2.556304,2.590351,5.691788,-3.321928,3.223401,...,4.084251,2.287523,3.205371,6.965451,-3.321928,4.531984,-3.321928,5.097337,1.895737,2.954997
6M1-18,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,...,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
7M1-2,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,...,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
7SK,-0.539253,-0.576620,-1.651323,0.126633,0.783715,-1.759556,-1.033968,-0.129513,0.494308,0.749841,...,0.233159,-0.026301,1.249039,1.651384,0.128604,-0.594062,-0.228405,0.239723,2.074377,0.867608


In [3]:
len(expression_df)

30865

In [4]:
series_matrix_1 = "GSE96058-GPL11154_series_matrix_selected_fixed_missing"
series_matrix_2 = "GSE96058-GPL18573_series_matrix_selected_fixed_missing"

series_matrix_df= pd.read_csv(f"Data/{series_matrix_1}.csv", index_col=0)  # Save as CSV
series_matrix_df_temp = pd.read_csv(f"Data/{series_matrix_2}.csv", index_col=0)  # Save as CSV
series_matrix_df = pd.concat([series_matrix_df, series_matrix_df_temp], axis=0)
del series_matrix_df_temp
len(series_matrix_df)
series_matrix_df.to_csv("Data/all_series_matrix.csv")

In [35]:
series_matrix_df['overall survival event'].value_counts()

overall survival event
0    3056
1     353
Name: count, dtype: int64

In [6]:
# check type and limits of the data
series_matrix_df.describe()

,age at diagnosis,tumor size,lymph node group,lymph node status,er status,pgr status,her2 status,ki67 status,nhg,pam50 subtype,overall survival days,overall survival event,endocrine treated,chemo treated
count,3409.000000,3374.000000,3409.000000,3409.000000,3341.000000,3317.000000,3403.000000,2913.000000,3409.000000,3409.000000,3409.000000,3409.000000,3387.000000,3388.000000
mean,62.738926,19.912567,1.314462,0.325022,0.874888,0.809466,0.113135,0.510470,1.179231,2.043708,1594.341742,0.103549,0.775317,0.404664
std,13.151961,12.184801,0.980876,0.527939,0.330896,0.392781,0.316805,0.499976,0.746960,1.005923,494.185586,0.304720,0.417435,0.490899
min,24.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,56.000000,0.000000,0.000000,0.000000
25%,53.000000,12.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,2.000000,1212.000000,0.000000,1.000000,0.000000
50%,64.000000,17.000000,2.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,2.000000,1596.000000,0.000000,1.000000,0.000000
75%,71.000000,24.000000,2.000000,1.000000,1.000000,1.000000,0.000000,1.000000,2.000000,3.000000,2004.000000,0.000000,1.000000,1.000000
max,96.000000,126.000000,3.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,4.000000,2474.000000,1.000000,1.000000,1.000000


In [7]:
# check for nans and infs
series_matrix_df.isna().sum().sum(), series_matrix_df.isin([float('inf'), float('-inf')]).sum().sum()


(np.int64(740), np.int64(0))

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Drop rows with missing values
cleaned_series_matrix_df = series_matrix_df.dropna()

#Let's do one step in two
#The expression is a numerical vector, the result it has to be a number
#Let's create a number that is, is only quimio, only endo, both or any

# Function to map values
def map_treatment(row):
    if row['endocrine treated'] == 1 and row['chemo treated'] == 0:
        return 1  # Only endocrine
    elif row['endocrine treated'] == 0 and row['chemo treated'] == 1:
        return 2  # Only chemo
    elif row['endocrine treated'] == 1 and row['chemo treated'] == 1:
        return 3  # Both
    else:
        return 0  # None

# Apply the function to create the 'treatment' column
cleaned_series_matrix_df['treatment'] = cleaned_series_matrix_df.apply(map_treatment, axis=1)



/tmp/ipykernel_143613/1063306493.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_series_matrix_df['treatment'] = cleaned_series_matrix_df.apply(map_treatment, axis=1)


In [9]:
cleaned_series_matrix_df

,age at diagnosis,tumor size,lymph node group,lymph node status,er status,pgr status,her2 status,ki67 status,nhg,pam50 subtype,overall survival days,overall survival event,endocrine treated,chemo treated,treatment
F1,43,9.0,2,0,0.0,0.0,0.0,1.0,2,0,2367,0,0.0,1.0,2
F2,48,14.0,0,1,1.0,1.0,0.0,0.0,1,2,2367,0,1.0,1.0,3
F3,69,27.0,1,1,1.0,1.0,0.0,1.0,2,3,2168,1,1.0,1.0,3
F4,39,51.0,0,1,1.0,1.0,1.0,1.0,2,2,2416,0,1.0,1.0,3
F5,73,60.0,1,1,1.0,1.0,0.0,0.0,1,4,2389,0,1.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
F3028repl,33,20.0,0,1,1.0,1.0,1.0,1.0,1,2,1251,0,1.0,1.0,3
F3058repl,85,23.0,1,1,1.0,1.0,0.0,1.0,1,3,1473,0,1.0,0.0,1
F3127repl,69,24.0,2,0,1.0,1.0,1.0,1.0,2,1,1277,0,1.0,1.0,3
F3250repl,42,15.0,2,0,0.0,0.0,1.0,1.0,2,1,904,0,0.0,1.0,2


In [20]:
# check class imbalance
treatment_counts = cleaned_series_matrix_df['treatment'].value_counts()
treatment_counts

treatment
1    1325
3     800
2     313
0     309
Name: count, dtype: int64

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression  # Or RandomForestClassifier, etc.
from sklearn.metrics import accuracy_score, classification_report
import pickle
from sklearn.metrics import matthews_corrcoef


def run_classification(df, features, target, test_size=0.2, random_state=42, save_model=True, model_filename="classification_model.pkl"):
    """
    Runs a classification model, evaluates it, and optionally saves the model.

    Args:
        df (pd.DataFrame): The input DataFrame.
        features (list): A list of column names to use as features.
        target (str): The column name of the target variable (categorical).
        test_size (float): The proportion of the data to use for the test set.
        random_state (int): Random seed for reproducibility.
        save_model (bool): Whether to save the trained model to a file.
        model_filename (str): The filename to use for saving the model.

    Returns:
        tuple: A tuple containing:
            - model: The trained classification model.
            - accuracy (float): The accuracy on the test set.
            - classification_report_str (str): The classification report.
    """

    # Separate features (X) and target (y)
    X = df[features]
    y = df[target]

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # Scale the features (important for many classifiers)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Create and train the classification model
    model = LogisticRegression(multi_class='ovr', solver='liblinear', class_weight='balanced')
    model.fit(X_train_scaled, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_scaled)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy:.2f}")
    mcc= matthews_corrcoef(y_test, y_pred)
    print(f"MCC: {mcc:.2f}")
    # Generate a classification report for more detailed evaluation
    classification_report_str = classification_report(y_test, y_pred)
    print("\nClassification Report:")
    print(classification_report_str)

    if save_model:
        try:
            pickle.dump(model, open(model_filename, "wb"))
            pickle.dump(scaler, open("scaler.pkl", "wb"))
            print(f"\nModel saved to {model_filename}")
        except Exception as e:
            print(f"Error saving model: {e}")

    return model, accuracy, classification_report_str

In [24]:
#1. Select columns to skip in the expression data
columns_to_skip = ['endocrine treated','chemo treated']
#2. Create a copy of the base dataframe
df =  cleaned_series_matrix_df.copy()
df= df.drop(columns=columns_to_skip, errors='ignore')
# sample 50 percent of the data
df_training = df.sample(frac=0.5, random_state=42)

#4.Define target and features, check which is the best.
target_variable = 'treatment' 
feature_columns = df_training.columns.tolist()[:-3]

#5. Use the defined function
model, mse, feature_importances = run_classification(
    df=df,
    features=feature_columns,
    target=target_variable,
    test_size=0.2,
    random_state=42,
    save_model=True,
    model_filename="treatment_prediction_model.pkl"
)
# get the rows on cleaned_series_matrix_df that are not in df_training



Accuracy: 0.73
MCC: 0.59

Classification Report:
              precision    recall  f1-score   support

           0       0.42      0.07      0.12        73
           1       0.73      0.86      0.79       246
           2       0.73      0.81      0.77        64
           3       0.75      0.79      0.77       167

    accuracy                           0.73       550
   macro avg       0.66      0.63      0.61       550
weighted avg       0.69      0.73      0.69       550


Model saved to treatment_prediction_model.pkl


/home/karen/Documents/GitHub/rnaseq-breast-cancer-ai/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1273: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/karen/Documents/GitHub/rnaseq-breast-cancer-ai/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


In [12]:
df_not_in_training = cleaned_series_matrix_df[~cleaned_series_matrix_df.index.isin(df_training.index)]
#6. Predict the treatment for the rows not in training
X_not_in_training = df_not_in_training[feature_columns]
X_not_in_training_scaled = StandardScaler().fit_transform(X_not_in_training)
# Make predictions
predictions = model.predict(X_not_in_training_scaled)
# Add predictions to the DataFrame
X_not_in_training['predicted_treatment'] = predictions
# evaluate the predictions
mse_eval = mean_squared_error(df_not_in_training[target_variable], predictions)
print(f"Mean Squared Error on not in training data: {mse_eval:.2f}")

Mean Squared Error on not in training data: 0.68


/tmp/ipykernel_143613/4019175284.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_not_in_training['predicted_treatment'] = predictions


In [13]:
matthews_corrcoef(df_not_in_training[target_variable], predictions)

0.6179859111243011

In [18]:
model.coef_

array([[ 0.69231496, -1.11448243,  0.04376743, -0.29918957, -0.55682608,
        -0.34565839, -0.34097776, -0.22158946, -0.45087024,  0.00921587],
       [ 1.0883153 ,  0.05554344,  0.01416733, -0.32735636,  1.22086315,
         0.26219113, -0.40981624, -0.56118034, -0.21541337, -0.05087093],
       [-0.86015324, -0.16883556, -0.04480358, -0.07728021, -1.05289124,
        -0.76251484,  0.23434013,  0.2816932 ,  0.23617868, -0.39983121],
       [-1.68330459,  0.38747241,  0.02765535,  0.78446487,  1.68941938,
         0.20484752,  0.51469   ,  0.88512205,  0.54294066,  0.24267974]])

In [16]:
feature_columns

['age at diagnosis',
 'tumor size',
 'lymph node group',
 'lymph node status',
 'er status',
 'pgr status',
 'her2 status',
 'ki67 status',
 'nhg',
 'pam50 subtype']

In [19]:
# for each elemnent on model.coef_, create the pandas series and then the pandas df of the coeficients ofr each feature
coef_df = pd.DataFrame(model.coef_, columns=feature_columns)
coef_df

,age at diagnosis,tumor size,lymph node group,lymph node status,er status,pgr status,her2 status,ki67 status,nhg,pam50 subtype
0,0.692315,-1.114482,0.043767,-0.299190,-0.556826,-0.345658,-0.340978,-0.221589,-0.450870,0.009216
1,1.088315,0.055543,0.014167,-0.327356,1.220863,0.262191,-0.409816,-0.561180,-0.215413,-0.050871
2,-0.860153,-0.168836,-0.044804,-0.077280,-1.052891,-0.762515,0.234340,0.281693,0.236179,-0.399831
3,-1.683305,0.387472,0.027655,0.784465,1.689419,0.204848,0.514690,0.885122,0.542941,0.242680


In [ ]:
expression_df

30865

In [28]:
series_matrix_df.index =  series_matrix_df.index.astype(str)
expression_df.index = expression_df.index.astype(str)

# Merge the dataframes
merged_df = series_matrix_df.merge(expression_df.T, left_index=True, right_index=True, how='inner')
merged_df.head()

,age at diagnosis,tumor size,lymph node group,lymph node status,er status,pgr status,her2 status,ki67 status,nhg,pam50 subtype,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
F1,43,9.0,2,0,0.0,0.0,0.0,1.0,2,0,...,1.108204,3.044573,1.640598,2.425306,3.244432,-0.411950,1.468899,6.281767,1.774107,2.437227
F2,48,14.0,0,1,1.0,1.0,0.0,0.0,1,2,...,1.678388,2.743902,0.684370,1.861781,2.727427,-0.452902,1.924761,7.169613,1.764214,2.876100
F3,69,27.0,1,1,1.0,1.0,0.0,1.0,2,3,...,2.443392,4.719843,-0.160076,1.365396,3.122333,0.594147,1.619277,6.683400,2.039246,2.929346
F4,39,51.0,0,1,1.0,1.0,1.0,1.0,2,2,...,2.441704,4.157466,1.036271,2.038984,3.182823,-0.046619,1.969433,6.666306,2.771650,2.777674
F5,73,60.0,1,1,1.0,1.0,0.0,0.0,1,4,...,1.749230,1.024820,1.147535,2.032704,3.588842,0.808823,2.998474,6.479006,2.696269,3.957146


In [29]:
len(merged_df)

3409

In [30]:
merged_df.dropna(inplace=True)
merged_df

,age at diagnosis,tumor size,lymph node group,lymph node status,er status,pgr status,her2 status,ki67 status,nhg,pam50 subtype,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
F1,43,9.0,2,0,0.0,0.0,0.0,1.0,2,0,...,1.108204,3.044573,1.640598,2.425306,3.244432,-0.411950,1.468899,6.281767,1.774107,2.437227
F2,48,14.0,0,1,1.0,1.0,0.0,0.0,1,2,...,1.678388,2.743902,0.684370,1.861781,2.727427,-0.452902,1.924761,7.169613,1.764214,2.876100
F3,69,27.0,1,1,1.0,1.0,0.0,1.0,2,3,...,2.443392,4.719843,-0.160076,1.365396,3.122333,0.594147,1.619277,6.683400,2.039246,2.929346
F4,39,51.0,0,1,1.0,1.0,1.0,1.0,2,2,...,2.441704,4.157466,1.036271,2.038984,3.182823,-0.046619,1.969433,6.666306,2.771650,2.777674
F5,73,60.0,1,1,1.0,1.0,0.0,0.0,1,4,...,1.749230,1.024820,1.147535,2.032704,3.588842,0.808823,2.998474,6.479006,2.696269,3.957146
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
F3028repl,33,20.0,0,1,1.0,1.0,1.0,1.0,1,2,...,2.001905,3.078963,0.944806,1.552461,2.981094,-0.822707,2.420330,6.660357,2.693414,3.245652
F3058repl,85,23.0,1,1,1.0,1.0,0.0,1.0,1,3,...,2.032610,3.853862,0.213366,1.635267,3.964512,-0.094584,1.623571,6.630378,1.367481,2.702757
F3127repl,69,24.0,2,0,1.0,1.0,1.0,1.0,2,1,...,2.526857,4.820546,0.164876,1.907848,4.372805,-1.244932,2.232271,6.553033,2.444790,4.057510
F3250repl,42,15.0,2,0,0.0,0.0,1.0,1.0,2,1,...,2.416890,2.458072,-0.097047,1.147157,2.851369,-1.073674,2.493925,6.209431,2.322594,3.277873


In [34]:
#1. Select columns to skip in the expression data
columns_to_skip = ['endocrine treated','chemo treated']
#2. Create a copy of the base dataframe
df =  merged_df.copy()

df['treatment'] = df.apply(map_treatment, axis=1)
df= df.drop(columns=columns_to_skip, errors='ignore')
# sample 50 percent of the data
df.to_csv("Data/merged_series_matrix_expression.csv", index=False)
df_training = df.sample(frac=0.5, random_state=42)

#4.Define target and features, check which is the best.
target_variable = 'treatment' 
feature_columns = df_training.columns.tolist()[:-3]



In [ ]:
#5. Use the defined function
model, mse, feature_importances = run_classification(
    df=df,
    features=feature_columns,
    target=target_variable,
    test_size=0.2,
    random_state=42,
    save_model=True,
    model_filename="treatment_prediction_model.pkl"
)
